In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ufal.udpipe
!wget https://github.com/jwijffels/udpipe.models.ud.2.5/raw/master/inst/udpipe-ud-2.5-191206/russian-syntagrus-ud-2.5-191206.udpipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.8/936.8 kB 11.7 MB/s eta 0:00:00
--2025-06-28 11:44:38--  https://github.com/jwijffels/udpipe.models.ud.2.5/raw/master/inst/udpipe-ud-2.5-191206/russian-syntagrus-ud-2.5-191206.udpipe
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/jwijffels/udpipe.models.ud.2.5/master/inst/udpipe-ud-2.5-191206/russian-syntagrus-ud-2.5-191206.udpipe [following]
--2025-06-28 11:44:38--  https://raw.githubusercontent.com/jwijffels/udpipe.models.ud.2.5/master/inst/udpipe-ud-2.5-191206/russian-syntagrus-ud-2.5-191206.udpipe
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Lengt

In [3]:
import os
from ufal.udpipe import Model, Pipeline

# **✅ Морфологический анализ корпуса**

In [5]:
# Инициализация модели UDPipe
model = Model.load("russian-syntagrus-ud-2.5-191206.udpipe")
if not model:
    raise Exception("Не удалось загрузить модель UDPipe!")
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

# Пути к вашим файлам
input_folder = "/content/drive/My Drive/SFU 3/Ex ml/2_морфологический_анализ_с_использованием_UDPipe/Lemma/Предобработанные"  # Папка с clean_*.txt
output_folder = "/content/drive/My Drive/SFU 3/Ex ml/2_морфологический_анализ_с_использованием_UDPipe/Lemma/Moprphological analysis"    # Для результатов
os.makedirs(output_folder, exist_ok=True)

# Обработка всех файлов
for filename in os.listdir(input_folder):
    if filename.startswith("clean_") and filename.endswith(".txt"):
        # Чтение файла
        with open(os.path.join(input_folder, filename), 'r', encoding='utf-8') as f:
            text = f.read()

        # Анализ через UDPipe
        processed = pipeline.process(text)

        # Сохранение результата
        output_filename = filename.replace("clean_", "udpipe_")
        with open(os.path.join(output_folder, output_filename), 'w', encoding='utf-8') as f:
            f.write(processed)

print("Обработка завершена! Результаты сохранены в:", output_folder)

Обработка завершена! Результаты сохранены в: /content/drive/My Drive/SFU 3/Ex ml/2_морфологический_анализ_с_использованием_UDPipe/Lemma/Moprphological analysis


# **✅ Анализ корпуса**

In [9]:
from collections import defaultdict # Импорт defaultdict для удобного подсчёта частот
import os

def calculate_pos_distribution_from_conllu(folder_path):
    pos_counts = defaultdict(int) # Создаём словарь для подсчёта количества каждой части речи
    total_words = 0

    for filename in os.listdir(folder_path):
        if filename.startswith("udpipe_") and filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip() # Убираем пробелы и символы перевода строки
                    if line and not line.startswith('#') and '\t' in line: # Пропускаем пустые строки и комментарии (# newdoc, # text, # sent_id и т.д.)
                        parts = line.split('\t') # Разбиваем строку по табуляции
                        if len(parts) > 3 and parts[3] != 'PUNCT': # Проверяем, что колонок достаточно и игнорируем пунктуацию
                            pos = parts[3]  # В 4 колонке (parts[3]) хранится UPOS (часть речи)
                            pos_counts[pos] += 1
                            total_words += 1

    print("Распределение частей речи:")
    for pos, count in sorted(pos_counts.items(), key=lambda x: x[1], reverse=True): # Сортируем по убыванию частоты и выводим процент и абсолютное значение
        print(f"{pos}: {count/total_words:.1%} ({count})")

    return pos_counts

pos_stats = calculate_pos_distribution_from_conllu(
    "/content/drive/My Drive/SFU 3/Ex ml/2_морфологический_анализ_с_использованием_UDPipe/Lemma/Moprphological analysis"
)


Распределение частей речи:
NOUN: 41.0% (24529)
VERB: 30.1% (17966)
ADJ: 11.6% (6956)
ADV: 7.0% (4182)
DET: 3.7% (2226)
PRON: 2.5% (1474)
NUM: 1.4% (859)
ADP: 0.8% (480)
PART: 0.6% (385)
SCONJ: 0.4% (231)
PROPN: 0.4% (223)
AUX: 0.2% (124)
INTJ: 0.1% (60)
X: 0.1% (42)
CCONJ: 0.1% (42)


# **✅ Разбор 3 предложений: UDPipe, mystem, spacy, pymorphy**

In [ ]:
!pip install spacy
!python -m spacy download ru_core_news_sm
import spacy

!pip install pymorphy2==0.8
from pymorphy2 import MorphAnalyzer

!wget http://download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz
!tar -xvf mystem-3.0-linux3.1-64bit.tar.gz
!cp mystem /bin
from pymystem3 import Mystem

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 38.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
--2025-06-27 22:12:38--  http://download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz
Resolving download.cdn.yandex.net (download.cdn.yandex.net)... 37.9.64.225, 2a02:6b8:23::225
Connecting to download.cdn.yandex.net (download.cdn.yandex.net)|37.9.64.225|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://cloudcdn-m9-9.cdn.yandex.net/download.cdn.yandex.net/mystem/mystem-3.0-linux3

In [ ]:
# Инициализация всех анализаторов
nlp = spacy.load("ru_core_news_sm")
mystem = Mystem()
pymorphy = MorphAnalyzer()


# Предложения для анализа
sample_sentences = [
    "Несколько, впрочем, странно и необыденно только одно: у северной двери стоит отец Григорий , еще не снимавший облачения, и сердито мигает своими густыми бровями.",
    "На стене в золотых рамках под стеклом висят письма какого-то петербургского гомеопата, по мнению Марфы Петровны , очень знаменитого и даже великого, и висит портрет отца Аристарха , которому генеральша обязана своим спасением: отречением от зловредной аллопатии и знанием истины.",
    "Во-первых, платить сто двадцать рублей за дачу для одной тяжело и, во-вторых, как-то жутко: вдруг вор заберется ночью или днем войдет страшный мужик!"
]

def analyze_with_udpipe(text):
#Анализ через UDPipe с полной морфологической разметкой
    processed = pipeline.process(text)
    return [line for line in processed.split('\n') if line and not line.startswith('#')]

def analyze_with_spacy(text):
#Анализ через spaCy
    doc = nlp(text)
    return [(token.text, token.lemma_, token.pos_, token.tag_) for token in doc]

def analyze_with_pymorphy(text):
#Анализ через pymorphy2
    words = text.split()
    return [(word, pymorphy.parse(word)[0].normal_form, str(pymorphy.parse(word)[0].tag)) for word in words]

def analyze_with_mystem(text):
#Анализ через MyStem
    analysis = mystem.analyze(text)
    results = []
    for word in analysis:
        if 'analysis' in word and word['analysis']:
            ana = word['analysis'][0]
            results.append((word['text'], ana['lex'], ana['gr']))
    return results

def print_comparison(sentence):
    print(f"\n\nПредложение\n{sentence}")

    print("\nUDPipe:")
    for line in analyze_with_udpipe(sentence):
        print(line)

    print("\nspaCy:")
    for token in analyze_with_spacy(sentence):
        print(f"{token[0]:<15} | {token[1]:<15} | {token[2]:<10} | {token[3]}")

    print("\npymorphy2:")
    for analysis in analyze_with_pymorphy(sentence):
        print(f"{analysis[0]:<15} | {analysis[1]:<15} | {analysis[2]}")

    print("\nMyStem:")
    for analysis in analyze_with_mystem(sentence):
        print(f"{analysis[0]:<15} | {analysis[1]:<15} | {analysis[2]}")

# Запуск сравнения для всех предложений
for sent in sample_sentences:
    print_comparison(sent)
    print("\n" + "="*80 + "\n")



Предложение
Несколько, впрочем, странно и необыденно только одно: у северной двери стоит отец Григорий , еще не снимавший облачения, и сердито мигает своими густыми бровями.

UDPipe:
1	Несколько	несколько	NUM	_	Animacy=Inan|Case=Acc	5	nsubj	_	SpaceAfter=No
2	,	,	PUNCT	_	_	1	punct	_	_
3	впрочем	впрочем	ADV	_	Degree=Pos	5	parataxis	_	SpaceAfter=No
4	,	,	PUNCT	_	_	3	punct	_	_
5	странно	странно	ADJ	_	Degree=Pos|Gender=Neut|Number=Sing|Variant=Short	0	root	_	_
6	и	и	CCONJ	_	_	7	cc	_	_
7	необыденно	необыденно	ADJ	_	Degree=Pos|Gender=Neut|Number=Sing|Variant=Short	5	conj	_	_
8	только	только	PART	_	_	9	advmod	_	_
9	одно	один	NUM	_	Case=Acc|Gender=Neut	5	nummod:gov	_	SpaceAfter=No
10	:	:	PUNCT	_	_	9	punct	_	_
11	у	у	ADP	_	_	13	case	_	_
12	северной	северный	ADJ	_	Case=Gen|Degree=Pos|Gender=Fem|Number=Sing	13	amod	_	_
13	двери	дверь	NOUN	_	Animacy=Inan|Case=Gen|Gender=Fem|Number=Sing	14	obl	_	_
14	стоит	стоять	VERB	_	Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act	9	p